# ViT Memory Autoencoder Training (Colab)

This notebook runs the rebuilt module-based pipeline using `train.py` and `evaluate.py`.

In [1]:
# Clone repository and install dependencies
import os

repo_dir = "/content/explainable-anomaly-detection"
if not os.path.isdir(repo_dir):
    !git clone https://github.com/eftekin/explainable-anomaly-detection {repo_dir}

%cd /content/explainable-anomaly-detection
!pip install -q -r requirements.txt
!pip install -q kaggle
print("Repository ready.")

/content/explainable-anomaly-detection
Repository ready.


In [ ]:
import os
import json

# Kaggle credentials
KAGGLE_USERNAME = "eftekin"
KAGGLE_KEY = ""   # empty — user fills this in

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
cred_path = os.path.expanduser('~/.kaggle/kaggle.json')
with open(cred_path, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(cred_path, 0o600)
print('Kaggle credentials saved to ~/.kaggle/kaggle.json')

Kaggle credentials saved to ~/.kaggle/kaggle.json


In [3]:
import os
import shutil
import subprocess

project_root = '/content/explainable-anomaly-detection'
if os.path.isdir(project_root):
    os.chdir(project_root)

mvtec_root = os.path.join('data', 'mvtec')
categories = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

if not os.path.exists(mvtec_root):
    os.makedirs(mvtec_root, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'ipythonx/mvtec-ad',
        '-p', '/content',
        '--unzip'
    ], check=True)
    for cat in categories:
        src = os.path.join('/content', cat)
        dst = os.path.join(mvtec_root, cat)
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.move(src, dst)

print('Dataset root:', mvtec_root)

Dataset root: data/mvtec


In [4]:
# Run training
!python train.py --data-root data/mvtec --category bottle

Device: cuda
Config | category=bottle | image=384 | tau=0.05 | lambda_ent=0.1 | freeze=50
Training 100 epochs | tau=0.05 | lambda_ent=0.1 | freeze_until=50
ep 001/100 | total=0.58970 | recon=0.19795 | ent=0.39176 | phase=1
ep 005/100 | total=0.12132 | recon=0.09723 | ent=0.02409 | phase=1
ep 010/100 | total=0.09557 | recon=0.08405 | ent=0.01152 | phase=1
  [Collapse] ep 10: variance=1.00e+00 [OK]
ep 015/100 | total=0.08883 | recon=0.07914 | ent=0.00969 | phase=1
ep 020/100 | total=0.08422 | recon=0.07568 | ent=0.00854 | phase=1
  [Collapse] ep 20: variance=9.80e-01 [OK]
ep 025/100 | total=0.08067 | recon=0.07285 | ent=0.00782 | phase=1
ep 030/100 | total=0.07798 | recon=0.07075 | ent=0.00723 | phase=1
  [Collapse] ep 30: variance=9.93e-01 [OK]
ep 035/100 | total=0.07603 | recon=0.06916 | ent=0.00687 | phase=1
ep 040/100 | total=0.07481 | recon=0.06818 | ent=0.00663 | phase=1
  [Collapse] ep 40: variance=9.35e-01 [OK]
ep 045/100 | total=0.07371 | recon=0.06723 | ent=0.00648 | phase=1
ep

In [5]:
# Run evaluation
!python evaluate.py --data-root data/mvtec --category bottle --checkpoint checkpoints/best_model.pth

Device: cuda
Evaluating: 100% 83/83 [00:03<00:00, 21.02it/s]
EVALUATION RESULTS
Category:          bottle
Image-level AUROC: 0.9667 (96.7%)
Pixel-level AUROC: 0.8791 (87.9%)
Top-k ratio:       0.1
Scoring direction: error up -> score up
Saved metrics to outputs/evaluation_metrics.json
